# Imports

In [1]:
import os
import torch
import numpy as np
import sys
import pandas as pd
from typing import Dict, Tuple, List
from torch import Tensor
from functools import cache
from torch_geometric.data import HeteroData
sys.path.append('../..')
from src.commands.graph_learning.main import Config, Trainer
from src.commands.feature_extraction.graph_utils import Graph, EdgeTypes, NodeTypes
from definitions import EXPERIMENTS_DIR

# Utils

In [2]:
def load_histories(exp_name : str) -> pd.DataFrame:

    histories_dir = os.path.join(EXPERIMENTS_DIR, exp_name, 'histories')
    histories = []

    for filename in os.listdir(histories_dir):
        filename = os.path.join(histories_dir, filename)
        df = pd.read_csv(filename)
        df['iteration'] = np.arange(len(df))
        histories.append(df)

    histories = pd.concat(histories)

    return histories

In [3]:
def load_results(exp_name : str) -> Dict[str, Dict[str, Tensor]]:

    results_dir = os.path.join(EXPERIMENTS_DIR, exp_name, 'results')
    results = {}

    for filename in os.listdir(results_dir):
        result_path = os.path.join(results_dir, filename)
        exp = filename.split('.')[0].replace('predictions_', '')
        results[exp] = torch.load(result_path, map_location='cpu', weights_only=False)

    return results

In [4]:
@cache
def load_data(experiment : str) -> Tuple[Graph, HeteroData]:

    config = Config(experiment=experiment)
    trainer = Trainer(experiment=config.experiment, config=config.training_config, configs=['Attn'])

    data, graph = trainer.load_data(config=config.training_config, return_raw=True)

    return graph, data

In [5]:
@cache
def load_model(experiment : str, config_name : str) -> Tuple[Graph, HeteroData, torch.nn.Module]:

    config = Config(experiment=experiment)
    trainer = Trainer(experiment=config.experiment, config=config.training_config, configs=['Attn'])

    data, graph = load_data(experiment=experiment)

    model_config = config.training_config.configs[config_name]
    model = trainer.create_model(config=model_config, data=graph)

    print(model.load_state_dict(torch.load(model_config.model_path, weights_only=False)))

    return model

In [6]:
def get_dataframes(graph : Graph) -> List[pd.DataFrame]:

    articles_df = pd.DataFrame(
        data=list(
            map(
                lambda x : { 'id' : x.id,  **x.properties }, 
                graph.get_nodes('Article').nodes_generator
            )
        )
    ).set_index('id')

    entities_df = pd.DataFrame(
        data=list(
            map(
                lambda x : { 'id' : x.id, **x.properties }, 
                graph.get_nodes('Entity').nodes_generator
            )
        )
    )
    
    triples_df = pd.DataFrame(data=list(map(lambda x : {
        "id" : x.id,
        "src" : x.src,
        "dst" : x.dst,
        "predicate" : x.properties['predicate']
    }, graph.get_edges(EdgeTypes.Entity_HASRELATIONWITH_Entity).edges_generator)))

    articles_df = articles_df \
        .join(pd.DataFrame(graph._edges[EdgeTypes.Article_HASKEYWORD_Keyword].edges_generator)[['src', 'dst']] \
        .groupby('src').agg(list)) \
        .rename(columns={'dst':'keywords'})

    return articles_df, entities_df, triples_df

In [7]:
def style_paper_plot(
    fig,
    *,
    width=950,
    height=600,
    font_family="Arial",
    font_size=14,
):
    fig.update_layout(
        width=width,
        height=height,
        font=dict(
            family=font_family,
            size=font_size,
        ),
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(
            l=80,
            r=30,
            t=30,
            b=70,
        ),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        ticks="outside",
        showline=True,
        linewidth=1,
        linecolor="black",
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor="lightgray",
        gridwidth=0.5,
        zeroline=False,
        ticks="outside",
        showline=True,
        linewidth=1,
        linecolor="black",
    )

    return fig

# Load History

In [8]:
EXPERIMENT = "topic_modeling/vqvae"
EXPERIMENT_PATH = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
config = Config(experiment=EXPERIMENT)

In [9]:
histories = load_histories(EXPERIMENT)
results = load_results(EXPERIMENT)

In [10]:
results.keys()

dict_keys(['baseline', 'baseline_128', 'baseline_vqpool', 'baseline_vqpool_8_16', 'baseline_8_16'])

In [11]:
histories.head()

,epoch,phase,loss,codebook_loss,commitement_loss,cosine_rec_loss__MAE,cosine_rec_loss__R2,cosine_rec_loss,config,iteration
0,0,train,0.717937,0.054157,0.054157,0.803620,-36.179016,0.663780,baseline_vqpool,0
1,0,train,0.256721,0.041511,0.041511,0.429635,-10.825094,0.215210,baseline_vqpool,1
2,0,train,0.171372,0.046049,0.046049,0.293624,-6.017670,0.125323,baseline_vqpool,2
3,0,train,0.279372,0.050481,0.050481,0.435208,-12.412649,0.228891,baseline_vqpool,3
4,0,train,0.220775,0.061255,0.061255,0.336028,-8.382977,0.159520,baseline_vqpool,4


In [12]:
aggregated = histories.drop(columns='iteration').groupby(by=['epoch', 'phase', 'config']).mean().reset_index()

In [13]:
mapping = {
    "baseline" : "codebook=None, hidden_dim=256",
    "baseline_128" : "codebook=(128,), hidden_dim=256, vqpool=False",
    "baseline_8_16" : "codebook=(8,16), hidden_dim=128, vqpool=False",
    "baseline_vqpool" : "codebook=(128,), hidden_dim=256, vqpool=True",
    "baseline_vqpool_8_16" : "codebook=(8,16), hidden_dim=128, vqpool=True",
}

In [34]:
results['baseline_8_16']['S'][:,8:].argmax(dim=-1).unique().shape

torch.Size([59])

In [35]:
def num_active_cluster(x):

    try:
        return len(results.get(x)['S'][:,-128:].argmax(dim=-1).unique())
    except Exception as e:
        return "/"

In [37]:
summary = aggregated[(aggregated['phase'] == 'val') & (aggregated['epoch'] == aggregated['epoch'].max())]
summary.loc[:,'Num. Active Clusters'] = summary['config'].apply(num_active_cluster)
summary.loc[:,'config'] = summary['config'].apply(mapping.get)
summary = summary.drop(columns=['epoch', 'phase'])
summary = summary.reindex()
summary.head()

/tmp/ipykernel_4798/2623196292.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary.loc[:,'Num. Active Clusters'] = summary['config'].apply(num_active_cluster)


,config,loss,codebook_loss,commitement_loss,cosine_rec_loss__MAE,cosine_rec_loss__R2,cosine_rec_loss,Num. Active Clusters
245,"codebook=None, hidden_dim=256",0.000334,0.000000,0.000000,0.013776,0.980194,0.000334,/
246,"codebook=(128,), hidden_dim=256, vqpool=False",0.018354,0.014067,0.014067,0.049384,0.747718,0.004288,44
247,"codebook=(8,16), hidden_dim=128, vqpool=False",0.020603,0.016610,0.016610,0.048123,0.765943,0.003992,59
248,"codebook=(128,), hidden_dim=256, vqpool=True",0.007997,0.000346,0.000346,0.067264,0.550185,0.007650,32
249,"codebook=(8,16), hidden_dim=128, vqpool=True",0.008300,0.000634,0.000634,0.067956,0.549475,0.007666,46
